In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 06 — Automatic Topic Interpretability Evaluation

Mengukur word intrusion, topic intrusion, semantic compactness,
dan inter-topic separation tanpa penilaian manual.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
%cd {PROJECT_ROOT}

%pip install -q \
    sentence-transformers \
    scikit-learn \
    pandas \
    numpy \
    openpyxl \
    pyarrow

In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

from src.hmrnet import HMRNet
from src.interpretability import (
    automatic_intruder_word_test,
    inter_topic_separation,
    semantic_compactness,
    topic_centroids,
    topic_intrusion_test,
)

TOP_N_WORDS = 10
model = HMRNet.load(
    CHECKPOINTS_DIR / "HMRNet-Full.pt",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
topics = model.get_topics(TOP_N_WORDS)
theta = model.transform(
    bow, semantic_embeddings, document_embeddings
)
dominant_topics = theta.argmax(axis=1)

encoder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cuda" if torch.cuda.is_available() else "cpu",
)


## 6.1 Automatic intruder-word test

In [ ]:
intruder_details, word_intrusion_score = (
    automatic_intruder_word_test(
        topics=topics,
        encoder=encoder,
        repetitions_per_topic=5,
        random_state=42,
    )
)
print("Word Intrusion Score (WIS):", word_intrusion_score)
display(intruder_details.head())


## 6.2 Topic intrusion test

In [ ]:
centroid_embeddings = topic_centroids(topics, encoder)
topic_intrusion_details, topic_intrusion_score = (
    topic_intrusion_test(
        document_embeddings=semantic_embeddings,
        topic_centroid_embeddings=centroid_embeddings,
        gold_topics=dominant_topics,
        candidate_count=4,
        random_state=42,
    )
)
print("Topic Intrusion Score (TIS):", topic_intrusion_score)
display(topic_intrusion_details.head())


## 6.3 Semantic compactness

In [ ]:
compactness_details, mean_compactness = semantic_compactness(
    topics,
    encoder,
)
print("Mean Semantic Compactness:", mean_compactness)
display(compactness_details)


## 6.4 Inter-topic separation

In [ ]:
separation = inter_topic_separation(topics, encoder)
print("Inter-topic Separation:", separation)


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Model": "HMRNet-Full",
            "Word_Intrusion_Score": word_intrusion_score,
            "Topic_Intrusion_Score": topic_intrusion_score,
            "Semantic_Compactness": mean_compactness,
            "Inter_Topic_Separation": separation,
        }
    ]
)

summary.to_csv(
    TABLES_DIR / "experiment_06_interpretability_summary.csv",
    index=False,
)
summary.to_excel(
    TABLES_DIR / "experiment_06_interpretability_summary.xlsx",
    index=False,
)
intruder_details.to_csv(
    TABLES_DIR / "experiment_06_word_intrusion_details.csv",
    index=False,
)
topic_intrusion_details.to_csv(
    TABLES_DIR / "experiment_06_topic_intrusion_details.csv",
    index=False,
)
compactness_details.to_csv(
    TABLES_DIR / "experiment_06_compactness_details.csv",
    index=False,
)

display(summary.round(4))


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.bar(
    compactness_details["Topic"].astype(str),
    compactness_details["Semantic_Compactness"],
)
plt.xlabel("Topic")
plt.ylabel("Semantic Compactness")
plt.title("Semantic Compactness by Topic")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_06_semantic_compactness.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()
